# MAF使用说明

## 环境准备

In [ ]:
#r "nuget:Microsoft.Agents.AI.OpenAI"

In [ ]:
const string API_KEY = "sk-d51699056d6c4f2a92fd71aba473d780";
const string END_POINT = "https://dashscope.aliyuncs.com/compatible-mode/v1";
const string MODEL_NAME = "qwen3.7-plus";
const string EMBEDDING_MODEL = "text-embedding-v4";

---

## 快速入门

In [ ]:
using Microsoft.Agents.AI;
using OpenAI;
using OpenAI.Chat;
using System.ClientModel;
var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
ChatClient chatClient = openAiClient.GetChatClient(MODEL_NAME);
ChatClientAgent agent = chatClient.AsAIAgent();
AgentResponse response = await agent.RunAsync("请介绍一下你自己");
Console.WriteLine(response);

---

## 流式输出

In [ ]:
using Microsoft.Agents.AI;
using OpenAI;
using OpenAI.Chat;
using System.ClientModel;

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
ChatClient chatClient = openAiClient.GetChatClient(MODEL_NAME);
ChatClientAgent agent = chatClient.AsAIAgent();

await foreach(var response in agent.RunStreamingAsync("请介绍一下你自己"))
{
    Console.WriteLine(response);
}

---

## 结构化输出

In [ ]:
using System.ComponentModel;
using System.Text.Json;
using System.Text.Json.Serialization;
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using OpenAI.Chat;
using ChatMessage = Microsoft.Extensions.AI.ChatMessage;

// 定义数据模型
[Description("Information about a city")]
public sealed class CityInfo
{
    [JsonPropertyName("name")]
    public string Name { get; set; }
}

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
ChatClient chatClient = openAiClient.GetChatClient(MODEL_NAME);
AIAgent agent = chatClient.AsAIAgent(new ChatClientAgentOptions()
{
    Name = "HelpfulAssistant",
    ChatOptions = new()
    {
        // 需要在提示词中声明返回的JSON格式
        Instructions = @"你是一个智能助手。请严格按照以下 JSON 格式输出，不要包含任何其他文字、解释或 Markdown 标记：
        {
            ""name"": ""城市名称""
        }",
        // OpenAI的可以通过下面的方式指定json_schema，不需要在提示词中手动指定
        //ResponseFormat = Microsoft.Extensions.AI.ChatResponseFormat.ForJsonSchema<CityInfo>()
    }
});

// 如果LLM返回的内容不是JSON，这里会出现异常
AgentResponse<CityInfo> response = await agent.RunAsync<CityInfo>("中国的首都是哪个城市？");

Console.WriteLine($"原始数据: {response.Text}");
Console.WriteLine($"城市名称: {response.Result.Name}");
Console.WriteLine();

错误示例代码，未在提示词中声明JSON格式时，会出现异常。

In [ ]:
using System.ComponentModel;
using System.Text.Json;
using System.Text.Json.Serialization;
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using OpenAI.Chat;
using ChatMessage = Microsoft.Extensions.AI.ChatMessage;

// 定义数据模型
[Description("Information about a city")]
public sealed class CityInfo
{
    [JsonPropertyName("name")]
    public string Name { get; set; }
}

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
ChatClient chatClient = openAiClient.GetChatClient(MODEL_NAME);
AIAgent agent = chatClient.AsAIAgent(new ChatClientAgentOptions()
{
    Name = "HelpfulAssistant",
    ChatOptions = new()
    {
        Instructions = @"你是一个智能助手。"
    }
});

AgentResponse<CityInfo> response = await agent.RunAsync<CityInfo>("中国的首都是哪个城市？");

Console.WriteLine($"原始数据: {response.Text}");
Console.WriteLine($"城市名称: {response.Result.Name}");
Console.WriteLine();

---

## 工具调用

通过`AIFunctionFactory`创建工具。

In [ ]:
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using OpenAI;
using OpenAI.Chat;
using System.ClientModel;

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
ChatClient chatClient = openAiClient.GetChatClient(MODEL_NAME);

PersonHelper personHelper = new PersonHelper();

ChatClientAgent agent = chatClient.AsAIAgent(
    instructions: "你是一个智能助手", 
    tools: 
    [
        AIFunctionFactory.Create(personHelper.Get, "获取人员的姓名")
    ]);
AgentResponse response = await agent.RunAsync("有多少个姓张的人？名称分别是什么？");
Console.WriteLine(response);

public class PersonHelper
{
    public string[] Get()
    {
        return ["张三", "张一", "张二", "李四", "王五"];
    }
}

对于工具的功能，可以使用`Description`特性来进行描述，这样更利于复用和维护。

In [ ]:
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using OpenAI;
using OpenAI.Chat;
using System.ClientModel;
using System.ComponentModel;

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
ChatClient chatClient = openAiClient.GetChatClient(MODEL_NAME);

PersonHelper personHelper = new PersonHelper();

ChatClientAgent agent = chatClient.AsAIAgent(
    instructions: "你是一个智能助手", 
    tools: 
    [
        AIFunctionFactory.Create(personHelper.Get)
    ]);
AgentResponse response = await agent.RunAsync("有多少个姓张的人？名称分别是什么？");
Console.WriteLine(response);

public class PersonHelper
{
    [Description("获取人员的姓名")]
    public string[] Get()
    {
        return ["张三", "张一", "张二", "李四", "王五"];
    }
}

如果两个地方都没有描述工具的功能，大模型不会进行工具的调用。

In [ ]:
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using OpenAI;
using OpenAI.Chat;
using System.ClientModel;
using System.ComponentModel;

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
ChatClient chatClient = openAiClient.GetChatClient(MODEL_NAME);

PersonHelper personHelper = new PersonHelper();

ChatClientAgent agent = chatClient.AsAIAgent(
    instructions: "你是一个智能助手", 
    tools: 
    [
        AIFunctionFactory.Create(personHelper.Get)
    ]);
AgentResponse response = await agent.RunAsync("有多少个姓张的人？名称分别是什么？");
Console.WriteLine(response);

public class PersonHelper
{
    public string[] Get()
    {
        return ["张三", "张一", "张二", "李四", "王五"];
    }
}

智能体`ChatClientAgent`可以使用方法`AsAIFunction`转换为工具，被其他的智能体调用。  
当使用这种方式时，需要在`description`参数里说明工具的用途。  
<span style="color: red;"> 使用 Polyglot Notebooks 调用时，并不是每次都会调用智能体工具。但是在 VS2022 中是每次都会调用的。</span>

In [ ]:
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using OpenAI;
using OpenAI.Chat;
using System.ClientModel;
using System.ComponentModel;

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
ChatClient chatClient = openAiClient.GetChatClient(MODEL_NAME);

PersonHelper personHelper = new PersonHelper();

ChatClientAgent personAgent = chatClient.AsAIAgent(
    instructions: "你是一个智能助手", 
    description: "获取人员的姓名",
    tools: 
    [
        AIFunctionFactory.Create(personHelper.Get)
    ]);

ChatClientAgent invokeAgent = chatClient.AsAIAgent(
    instructions: "你是一个智能助手",
    tools:
    [
        personAgent.AsAIFunction()
    ]);

var response = await invokeAgent.RunAsync("请帮我获取人员的姓名");
Console.WriteLine(response.Text);

public class PersonHelper
{
    public string[] Get()
    {
        return ["张三", "张一", "张二", "李四", "王五"];
    }
}

将MCP作为工具调用。

In [ ]:
#r "nuget:ModelContextProtocol,1.4.0"

In [ ]:
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using ModelContextProtocol.Client;
using OpenAI;
using OpenAI.Chat;
using System.ClientModel;

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
ChatClient chatClient = openAiClient.GetChatClient(MODEL_NAME);

McpClient mcpClient = await McpClient.CreateAsync(new HttpClientTransport(new HttpClientTransportOptions
{
    Endpoint = new Uri("https://learn.microsoft.com/api/mcp"),
    TransportMode = HttpTransportMode.StreamableHttp
}));
IList<McpClientTool> mcpTools = await mcpClient.ListToolsAsync();

foreach (var item in mcpTools)
{
    Console.WriteLine(item);
}


没有使用MCP工具时的输出示例。

In [ ]:
ChatClientAgent agentNoTool = chatClient.AsAIAgent(instructions: "你是一个智能助手");
await foreach(var response in agentNoTool.RunStreamingAsync("基于MAF框架，用C#写一个Agent的简单调用示例"))
{
    if (string.IsNullOrEmpty(response.Text))
    {
        continue;
    }

    Console.Write(response.Text);
}

使用MCP工具时的输出示例。

In [ ]:
ChatClientAgent agentHaveTool = chatClient.AsAIAgent(
    instructions: "你是一个智能助手",
    tools: mcpTools.Cast<AITool>().ToList());
await foreach(var response in agentHaveTool.RunStreamingAsync("基于MAF框架，用C#写一个Agent的简单调用示例"))
{
    if (string.IsNullOrEmpty(response.Text))
    {
        continue;
    }

    Console.Write(response.Text);
}

---

## 大模型调用详情

可以查看Token用量，请求和答复的原始数据，也可以直接篡改数据。

In [ ]:
using Microsoft.Agents.AI;
using OpenAI;
using OpenAI.Chat;
using System.ClientModel;
using System.Net.Http;
using System.Text.Json;
using System.ClientModel.Primitives;
using System.Threading;

class CustomClientHttpHandler : HttpClientHandler
{
    protected override async Task<HttpResponseMessage> SendAsync(HttpRequestMessage request, CancellationToken cancellationToken)
    {
        string requestString = await (request.Content?.ReadAsStringAsync(cancellationToken) ?? Task.FromResult<string>(string.Empty));

        Console.WriteLine($"Raw Request ({request.RequestUri})");
        Console.WriteLine(MakePretty(requestString));
        Console.WriteLine(new string('-', 50));

        var response = await base.SendAsync(request, cancellationToken);
        string responseString = await response.Content.ReadAsStringAsync(cancellationToken);
        Console.WriteLine("Raw Response");
        Console.WriteLine(MakePretty(responseString));
        Console.WriteLine(new string('-', 50));

        return response;
    }

    private string MakePretty(string input)
    {
        if (string.IsNullOrEmpty(input)) return input;

        try
        {
            var jsonElement = JsonSerializer.Deserialize<JsonElement>(input);
            return JsonSerializer.Serialize(jsonElement, new JsonSerializerOptions { WriteIndented = true });
        }
        catch (Exception)
        {
            // 如果不是 JSON 格式，直接返回原文本
            return input;
        }
    }
}

public sealed class CityInfo
{
    public string Name { get; set; }
}

CustomClientHttpHandler handler = new CustomClientHttpHandler();
HttpClient client = new HttpClient(handler);
var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT),
    Transport = new HttpClientPipelineTransport(client)
});
ChatClient chatClient = openAiClient.GetChatClient(MODEL_NAME);
AIAgent agent = chatClient.AsAIAgent(new ChatClientAgentOptions()
{
    Name = "HelpfulAssistant",
    ChatOptions = new()
    {
        // 需要在提示词中声明返回的JSON格式
        Instructions = @"你是一个智能助手。请严格按照以下 JSON 格式输出，不要包含任何其他文字、解释或 Markdown 标记：
        {
            ""name"": ""城市名称""
        }",
        // OpenAI的可以通过下面的方式指定json_schema，不需要在提示词中手动指定
        //ResponseFormat = Microsoft.Extensions.AI.ChatResponseFormat.ForJsonSchema<CityInfo>()
    }
});

// 如果LLM返回的内容不是JSON，这里会出现异常
AgentResponse<CityInfo> response = await agent.RunAsync<CityInfo>("中国的首都是哪个城市？");

Console.WriteLine($"原始数据: {response.Text}");
Console.WriteLine($"城市名称: {response.Result.Name}");
Console.WriteLine();



---

## 增强检索（RAG）

### 嵌入和向量

In [ ]:
using OpenAI;
using OpenAI.Chat;
using OpenAI.Embeddings;
using System.ClientModel;
using Microsoft.Extensions.AI;

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
ChatClient chatClient = openAiClient.GetChatClient(MODEL_NAME);

EmbeddingClient embeddingClient =  openAiClient.GetEmbeddingClient(EMBEDDING_MODEL);
IEmbeddingGenerator<string, Embedding<float>> embeddingGenerator = embeddingClient.AsIEmbeddingGenerator();
Embedding<float> embeddingData = await embeddingGenerator.GenerateAsync("MAF框架介绍");
Console.WriteLine("向量长度：" + embeddingData.Vector.Length);
Console.WriteLine(string.Join('\n', embeddingData.Vector.ToArray()));

- 尺寸越小定义越模糊
- 尺寸越大需要压缩数据并丢失部分定义

### 向量存储

In [ ]:
#r "nuget:Microsoft.Extensions.AI.Abstractions, 10.6.0"
#r "nuget:Microsoft.SemanticKernel.Connectors.SqliteVec,1.74.0-preview"
#r "nuget:Microsoft.Agents.AI.OpenAI"

In [ ]:
const string fish = "鱼类是最古老的脊椎动物，几乎栖居于地球所有水域。它们终生生活在水中，用鳃呼吸，用鳍辅助运动与平衡，多为变温动物。鱼类种类繁多，不仅富含优质蛋白质，还包含众多色彩艳丽的观赏品种，在生态与人类生活中具有重要价值。";
const string dog = "狗类由早期人类从灰狼驯化而来，是人类最早驯养的家畜。它们机智勇敢、忠实可靠，拥有高度发达的嗅觉和听觉。狗的品种繁多，用途广泛，除了作为宠物陪伴人类，还常被用作导盲、搜救、牧羊和军警等工作犬，被誉为“人类最忠实的朋友”。";
const string cat = "猫类属于猫科猫属的小型食肉哺乳动物。它们体态轻盈，足底有肉垫，行走无声且极具攀爬本领。猫生性独立，有强烈的领地观念，多为夜行性。作为人类常见的宠物，猫不仅能捕鼠，还能提供情感陪伴，其丰富的面部表情和独特的叫声是长期与人类相处的进化结果。";
const string ragdollCat = "布偶猫：布偶猫以其如丝般柔顺的长毛和湛蓝如海的眼眸闻名，性格温顺得像一团柔软的云朵。它们体型较大，四肢粗壮，尾巴蓬松如狐尾，行走时优雅从容。布偶猫极度亲人，常被主人抱起时便全身放松，宛如布偶般瘫软，故得此名。它们对疼痛忍耐度高，极少伸爪，是理想的家庭伴侣猫。毛发虽长，但底层绒毛少，不易打结，定期梳理即可保持光泽。布偶猫好奇心强，喜欢跟随主人，却从不吵闹，用安静的陪伴诠释温柔。";
const string siameseCat = "暹罗猫：暹罗猫是猫中“狗派”，活泼好动，忠诚黏人，常像小狗般迎接主人归家。其标志性的重点色——面部、耳朵、四肢与尾巴呈深褐或灰蓝，与奶油色身躯形成鲜明对比，宛如精心绘制的艺术品。眼睛呈杏仁状，湛蓝深邃，仿佛藏着星辰。暹罗猫智商极高，可学会开门、捡球，甚至用爪子轻拍表达需求。它们爱“说话”，叫声清亮多变，常与主人对话。虽外表优雅，却精力旺盛，需充足玩具与互动，否则易因无聊而搞破坏。";
const string maineCoonCat = "缅因猫：缅因猫是猫界“温柔巨人”，成年体重可达10公斤以上，体长逾一米，却性格憨厚如犬。它们拥有浓密防水的被毛、蓬松大尾巴与耳尖标志性的“猞猁毛”，眼神沉稳，透着古老森林的智慧。缅因猫不喜高处，偏爱地面活动，常安静卧于主人脚边，用呼噜声传递安心。它们对水不抗拒，甚至喜欢玩水龙头，是少数能接受洗澡的猫种。虽体型庞大，动作却轻盈，跳跃无声。缅因猫包容性强，与狗、孩童皆能和睦共处，是家庭的守护者。";
const string scottishFoldCat = "苏格兰折耳猫：苏格兰折耳猫最显著特征是向前折叠的耳朵，源于基因突变，赋予其“猫头鹰般”的圆润脸庞与无辜眼神。它们体型中等，骨骼粗壮，尾巴短而灵活，行走时如小熊猫般憨态可掬。折耳猫性格极度温顺，安静内敛，极少跳跃或攀爬，常以“农民揣”姿势静卧，仿佛一尊毛绒雕塑。它们情感细腻，对主人依赖性强，却从不强求关注。需特别注意：折耳基因伴随软骨发育问题，需定期体检，避免剧烈运动，以温柔呵护其脆弱骨骼。";
const string sphynxCat = "斯芬克斯猫：斯芬克斯猫无毛或覆极短绒毛，皮肤褶皱如古埃及雕像，触感如温热丝绸，体温高于普通猫，是天然的“暖手宝”。它们耳朵硕大，眼睛呈柠檬形，眼神锐利却充满好奇，仿佛能洞察人心。斯芬克斯猫极度聪明，情感丰富，对主人忠诚至深，常如影子般跟随，甚至能感知情绪变化。因无毛，需保暖防晒，冬季穿毛衣，夏季避强光。它们爱干净，会主动舔舐皮肤，但需定期清洁褶皱防油脂堆积。虽外表奇特，内心却是最柔软的依恋。";

In [ ]:
using Microsoft.Extensions.AI;
using Microsoft.Extensions.VectorData;
using Microsoft.SemanticKernel.Connectors.SqliteVec;
using OpenAI;
using OpenAI.Chat;
using OpenAI.Embeddings;
using System.ClientModel;
using System.IO;

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
ChatClient chatClient = openAiClient.GetChatClient(MODEL_NAME);

EmbeddingClient embeddingClient =  openAiClient.GetEmbeddingClient(EMBEDDING_MODEL);
IEmbeddingGenerator<string, Embedding<float>> embeddingGenerator = embeddingClient.AsIEmbeddingGenerator();

string dataSource = "Data Source=Test.db";
File.Delete("Test.db");
var vectorStore = new SqliteVectorStore(dataSource, new SqliteVectorStoreOptions()
{
    EmbeddingGenerator = embeddingGenerator
});

List<KnowledgeRecord> list = new List<KnowledgeRecord>()
{
    new KnowledgeRecord()
    {
        Type = "鱼类",
        Description = fish
    },
    new KnowledgeRecord()
    {
        Type = "狗类",
        Description = dog
    },
    new KnowledgeRecord()
    {
        Type = "猫类",
        Description = cat
    },
    new KnowledgeRecord()
    {
        Type = "布偶猫",
        Description = ragdollCat
    },
    new KnowledgeRecord()
    {
        Type = "暹罗猫",
        Description = siameseCat
    },
    new KnowledgeRecord()
    {
        Type = "缅因猫",
        Description = maineCoonCat
    },
    new KnowledgeRecord()
    {
        Type = "苏格兰折耳猫",
        Description = scottishFoldCat
    },
    new KnowledgeRecord()
    {
        Type = "斯芬克斯猫",
        Description = sphynxCat
    }
};

var db = vectorStore.GetCollection<string, KnowledgeRecord>("Animal");
await db.EnsureCollectionExistsAsync();
await db.UpsertAsync(list);
Console.WriteLine("向量化数据库生成成功");

public class KnowledgeRecord
{
    public KnowledgeRecord()
    {

    }

    [VectorStoreKey]
    public required string Type { get; set; }

    [VectorStoreData]
    public required string Description { get; set; }

    [VectorStoreVector(1024)]
    public string Embedding => Description;

    [VectorStoreData]
    public string UpdateAt { get; set; } = DateTime.Now.ToString("yyyy-MM-dd HH:mm:ss");
}

**数据模型特性说明**
- VectorStoreKey
标记为主键，一个数据模型必须有这个的声明，不支持多个。
- VectorStoreVector
标记为向量数据，有两个参数：Dimensions-维度，DistanceFunction-距离计算方法。
这个特性在数据模型中也必须至少有一个。
- VectorStoreData
标记为普通数据，可有可无。这一类数据不会进行向量化，最普通的数据。

**向量数据库使用注意事项**
- 主键类型需要保持一致
vectorStore.GetCollection<TKey, TType>(name)，TKey的类型需要与TType中VectorStoreKey标记的类型一致。
- 向量维度需要保持一致
向量模型返回的维度，需要和VectorStoreVector标记的维度一致。
注意，这个维度是**不支持变更**，要变更时，需要重建表。
例如，以前是1024维，现在需要扩展为1536维，是不行的。
- 数据模型必须包含无参构造器
- VectorStoreVector标记的属性是无法查询数据的
如果要保留原始数据，则需要单独建一个属性。

### 向量检索

简单查询方式。

In [ ]:
await foreach(var item in db.GetAsync(record => true, int.MaxValue))
{
    Console.WriteLine("-------------------------------------------------");
    Console.WriteLine($"[{item.Type}][{item.UpdateAt}]");
    Console.WriteLine(item.Description);
    Console.WriteLine("-------------------------------------------------");
}

向量检索。

In [ ]:
await foreach (VectorSearchResult<KnowledgeRecord> item in db.SearchAsync("橘猫", 3))
{
    Console.WriteLine($"[{item.Score}]{item.Record.Description}");
}

根据使用的距离计算公式不同，分值所表示的含义也不一样。有的算法是分值越小越相似，有的则是越大越相似。
实际使用不需要关心这个，并且也不能完全根据分值判断是否为最相似的。

作为工具调用。

In [ ]:
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using Microsoft.Extensions.VectorData;
using Microsoft.SemanticKernel.Connectors.SqliteVec;
using OpenAI;
using OpenAI.Chat;
using OpenAI.Embeddings;
using System.ClientModel;

SearchTool searchTool = new SearchTool(db);
var toolAgent = chatClient.AsAIAgent(options: new ChatClientAgentOptions()
{
    ChatOptions = new ChatOptions()
    {
        Tools = [AIFunctionFactory.Create(searchTool.Search, "search_knowledge")]
    }
});

var response = await toolAgent.RunAsync("哪种猫的毛最柔软？");
Console.WriteLine(response.Text);

private class SearchTool
{
    public SearchTool(VectorStoreCollection<string, KnowledgeRecord> db) 
    {
        this.db = db;
    }

    private readonly VectorStoreCollection<string, KnowledgeRecord> db;

    public async Task<string> Search(string input)
    {
        StringBuilder sb = new StringBuilder();
        await foreach (VectorSearchResult<KnowledgeRecord> item in db.SearchAsync(input, 3))
        {
            sb.AppendLine(item.Record.Description);
        }

        return sb.ToString();
    }
}

---

## 多模态

识别图片内容。

In [ ]:
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using OpenAI;
using OpenAI.Chat;
using System.ClientModel;
using ChatMessage = Microsoft.Extensions.AI.ChatMessage;
using System.IO;

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
ChatClient chatClient = openAiClient.GetChatClient(MODEL_NAME);

string data = Convert.ToBase64String(File.ReadAllBytes("C:\\Users\\13009\\Desktop\\AIDemo\\什么时候用Spec.png"));
string dataUri = $"data:image/jpeg;base64,{data}";

ChatClientAgent agent = chatClient.AsAIAgent();
var response = await agent.RunAsync(new ChatMessage(ChatRole.User,
[
    new TextContent("描述一下图片的内容"),
    new DataContent(dataUri, "image/jpeg")
]));
Console.WriteLine(response);

识别PDF内容。<span style="color: red;">千问模型不支持这种数据输入，可能只有GPT可以。</span>仅作为了解。

In [ ]:
byte[] data = File.ReadAllBytes(path);
RunAsync(new ChatMessage(ChatRole.User, 
[
    new TextContent(""),
    new DataContent(data, "application/pdf")
]
));

---

## 上下文提供

In [ ]:
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using OpenAI;
using OpenAI.Chat;
using System.ClientModel;
using System.Threading;

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
ChatClient chatClient = openAiClient.GetChatClient(MODEL_NAME);

ChatClientAgent agent = chatClient.AsAIAgent(options: new ChatClientAgentOptions()
{
    AIContextProviders = [new CustomAIContextProvider()]
});
var response = await agent.RunAsync("你是谁？");
Console.WriteLine(response);

class CustomAIContextProvider : AIContextProvider
{
    private List<string> historyMessages = [];

    protected override ValueTask<AIContext> InvokingCoreAsync(InvokingContext context, CancellationToken cancellationToken = default)
    {
        return new ValueTask<AIContext>(new AIContext()
        {
            Instructions = "我是海绵宝宝。"
        });
    }

    protected override ValueTask<AIContext> ProvideAIContextAsync(InvokingContext context, CancellationToken cancellationToken = default)
    {
        return base.ProvideAIContextAsync(context, cancellationToken);
    }

    protected override ValueTask InvokedCoreAsync(InvokedContext context, CancellationToken cancellationToken = default)
    {
        return ValueTask.CompletedTask;
    }

    protected override ValueTask StoreAIContextAsync(InvokedContext context, CancellationToken cancellationToken = default)
    {
        return base.StoreAIContextAsync(context, cancellationToken);
    }
}

`InvokingCoreAsync`和`InvokedCoreAsync`没有调用基类的方法时，`ProvideAIContextAsync`和`StoreAIContextAsync`就不会触发了。  
`InvokingCoreAsync`用于提供额外的上下文给Agent。  
`InvokedCoreAsync`用于将AI回复的作为上下文。  

一个Agent，可以传入多个 `AIContextProvider`，相当于可以外挂多个知识库。  
**注意**，会按照初始化的顺序，依次调用上下文，只有最后一个返回的内容，会被传递给Agent。

In [ ]:
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using OpenAI;
using OpenAI.Chat;
using System.ClientModel;

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
ChatClient chatClient = openAiClient.GetChatClient(MODEL_NAME);

ChatClientAgent agent = chatClient.AsAIAgent(options: new ChatClientAgentOptions()
{
    AIContextProviders = [new CustomAIContextProvider(), new Custom1AIContextProvider()]
});

AgentSession session = await agent.CreateSessionAsync();
session.StateBag.SetValue("SessionId", Guid.NewGuid().ToString());

var response = await agent.RunAsync("你是谁？直接说名字，不要说多余的内容", session);
Console.WriteLine(response);

Console.WriteLine("------------------");

response = await agent.RunAsync("我是谁？直接说名字，不要说多余的内容", session);
Console.WriteLine(response);

class CustomAIContextProvider : AIContextProvider
{
    protected override ValueTask<AIContext> InvokingCoreAsync(InvokingContext context, CancellationToken cancellationToken = default)
    {
        return new ValueTask<AIContext>(new AIContext()
        {
            Instructions = "",
            Messages = [
                new Microsoft.Extensions.AI.ChatMessage(ChatRole.System, "你是海绵宝宝"),
            ]
        });
    }

    protected override ValueTask InvokedCoreAsync(InvokedContext context, CancellationToken cancellationToken = default)
    {
        return ValueTask.CompletedTask;
    }
}

class Custom1AIContextProvider : AIContextProvider
{
    protected override ValueTask<AIContext> InvokingCoreAsync(InvokingContext context, CancellationToken cancellationToken = default)
    {
        return new ValueTask<AIContext>(new AIContext()
        {
            Instructions = "",
            Messages = [
                new Microsoft.Extensions.AI.ChatMessage(ChatRole.User, "我是痞老板"),
            ]
        });
    }

    protected override ValueTask InvokedCoreAsync(InvokedContext context, CancellationToken cancellationToken = default)
    {
        return ValueTask.CompletedTask;
    }
}

In [ ]:
var rawJson = await agent.SerializeSessionAsync(session);
var options = new System.Text.Json.JsonSerializerOptions
{
    WriteIndented = true,
    Encoder = System.Text.Encodings.Web.JavaScriptEncoder.UnsafeRelaxedJsonEscaping
};
var doc = System.Text.Json.JsonDocument.Parse(rawJson.ToString());
string formattedJson = System.Text.Json.JsonSerializer.Serialize(doc, options);
Console.WriteLine(formattedJson);

## 历史会话存储

In [ ]:
#r "nuget:Microsoft.Extensions.AI.Abstractions, 10.6.0"
#r "nuget:Microsoft.SemanticKernel.Connectors.InMemory,1.74.0-preview"
#r "nuget:Microsoft.Agents.AI.OpenAI"

通过 `AIContextProvider` 实现历史会话存储。  
虽然可以，但是不符合这个接口的用途。

In [ ]:
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using OpenAI;
using OpenAI.Chat;
using System.ClientModel;
using Microsoft.SemanticKernel.Connectors.InMemory;
using Microsoft.Extensions.VectorData;

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
var chatClient = openAiClient.GetChatClient(MODEL_NAME);

VectorStore vectorStore = new InMemoryVectorStore(new InMemoryVectorStoreOptions()
{
    EmbeddingGenerator = openAiClient
        .GetEmbeddingClient(EMBEDDING_MODEL)
        .AsIEmbeddingGenerator()
});

ChatClientAgent agent = chatClient.AsAIAgent(options: new ChatClientAgentOptions()
{
    AIContextProviders = [new ChatHistoryMemoryProvider(
            vectorStore,
            collectionName: "chathistory",
            vectorDimensions: 1024,
            session => new ChatHistoryMemoryProvider.State(
                storageScope: new() { UserId = "UID1", SessionId = Guid.NewGuid().ToString() },
                searchScope: new() { UserId = "UID1" }))]
});

AgentSession session = await agent.CreateSessionAsync();
var response = await agent.RunAsync("我是海绵宝宝，我的好朋友是派大星。", session);
Console.WriteLine(response);

Console.WriteLine("------------------------------------------------");

response = await agent.RunAsync("我是谁？我的好朋友是谁？", session);
Console.WriteLine(response);

通过 `ChatHistoryProvider` 实现历史会话存储。

In [ ]:
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using OpenAI;
using OpenAI.Chat;
using System.ClientModel;
using System.Threading;

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
var chatClient = openAiClient.GetChatClient(MODEL_NAME);

ChatClientAgent agent = chatClient.AsAIAgent(options: new ChatClientAgentOptions()
{
    ChatHistoryProvider = new CustomChatHistoryProvider()
});

AgentSession session = await agent.CreateSessionAsync();
session.StateBag.SetValue("SessionId", Guid.NewGuid().ToString());

var response = await agent.RunAsync("我是海绵宝宝。", session);
Console.WriteLine(response);

Console.WriteLine("--------------------");

response = await agent.RunAsync("我的好朋友是派大星。", session);
Console.WriteLine(response);

Console.WriteLine("--------------------");

agent = chatClient.AsAIAgent(options: new ChatClientAgentOptions()
{
    ChatHistoryProvider = new CustomChatHistoryProvider()
});
response = await agent.RunAsync("我是谁？", session);
Console.WriteLine(response);

class CustomChatHistoryProvider : ChatHistoryProvider
{
    private string? sessionId = "";
    private static readonly Dictionary<string, List<Microsoft.Extensions.AI.ChatMessage>> messages = [];

    protected override ValueTask<IEnumerable<Microsoft.Extensions.AI.ChatMessage>> ProvideChatHistoryAsync(InvokingContext context, CancellationToken cancellationToken = default)
    {
        if (context.Session!.StateBag.TryGetValue("SessionId", out string? value))
        {
            sessionId = value;
        }
        else
        {
            sessionId = Guid.NewGuid().ToString();
            context.Session!.StateBag.SetValue("SessionId", sessionId);
        }

        if (!messages.ContainsKey(sessionId))
        {
            messages.Add(sessionId, new List<Microsoft.Extensions.AI.ChatMessage>());
        }

        return new ValueTask<IEnumerable<Microsoft.Extensions.AI.ChatMessage>>(messages[sessionId]);
    }

    protected override ValueTask StoreChatHistoryAsync(InvokedContext context, CancellationToken cancellationToken = default)
    {
        if (context.Session!.StateBag.TryGetValue("SessionId", out string? value))
        {
            sessionId = value;
        }
        else
        {
            sessionId = Guid.NewGuid().ToString();
            context.Session!.StateBag.SetValue("SessionId", sessionId);
        }

        if (!messages.ContainsKey(sessionId))
        {
            messages.Add(sessionId, new List<Microsoft.Extensions.AI.ChatMessage>());
        }

        messages[sessionId].AddRange(context.RequestMessages);
        messages[sessionId].AddRange(context.ResponseMessages ?? []);

        return ValueTask.CompletedTask;
    }
}

如果不需要持久化记忆，那就不需要传入 `ChatHistoryProvider`。

In [ ]:
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using OpenAI;
using OpenAI.Chat;
using System.ClientModel;
using System.Threading;

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
var chatClient = openAiClient.GetChatClient(MODEL_NAME);

ChatClientAgent agent = chatClient.AsAIAgent();

AgentSession session = await agent.CreateSessionAsync();
session.StateBag.SetValue("SessionId", Guid.NewGuid().ToString());

var response = await agent.RunAsync("我是海绵宝宝。", session);
Console.WriteLine(response);

Console.WriteLine("--------------------");

response = await agent.RunAsync("我的好朋友是派大星。", session);
Console.WriteLine(response);

Console.WriteLine("--------------------");

response = await agent.RunAsync("我是谁？", session);
Console.WriteLine(response);

使用 `AgentSession`时，一定需要设置 `SessionId`，不设置时，会出现异常。  
也可以通过 `AgentSession` 在初始化时就设置上下文内容。

In [ ]:
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;
using OpenAI;
using OpenAI.Chat;
using System.ClientModel;
using System.Threading;

var openAiClient = new OpenAIClient(new ApiKeyCredential(API_KEY), new OpenAIClientOptions()
{
    Endpoint = new Uri(END_POINT)
});
var chatClient = openAiClient.GetChatClient(MODEL_NAME);

ChatClientAgent agent = chatClient.AsAIAgent();

AgentSession session = await agent.CreateSessionAsync();
session.StateBag.SetValue("SessionId", Guid.NewGuid().ToString());

session.SetInMemoryChatHistory([
    new Microsoft.Extensions.AI.ChatMessage(ChatRole.User, "我是海绵宝宝。"),
    new Microsoft.Extensions.AI.ChatMessage(ChatRole.User, "我的好朋友是派大星。"),
    new Microsoft.Extensions.AI.ChatMessage(ChatRole.System, "你是章鱼哥"),
]);

var response = await agent.RunAsync("我是谁？你是谁？", session);
Console.WriteLine(response);

AgentSession可以序列化和反序列：agent.SerializeSessionAsync(session)和agent.DeserializeSessionAsync(json)